# 02 — Data and training

How the network of the adopted pipeline was trained: the split, the training samples, the model, the loss and the training
run. The training itself needs a GPU (it ran on a Kaggle T4): this notebook shows the code and the saved training log, and
retraining is behind a switch (`RUN_TRAINING`). Runs on CPU in a few minutes. Needs the cleaned masks created by notebook 01.



| data | use |
|---|---|
| 169 training images | training the network; training the object classifier (notebook 03) | 
| 43 validation images | choosing the checkpoint (best epoch); checking every choice | 
| 5-fold cross-validation on the 212 trainval images | 10 extra networks to compare training variations and post-processing (`experiments/`) | 
| 70 test images | final evaluation only (notebook 04) |

In [ ]:
import json
import os
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from skimage import measure

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
from models.c_res_unet_small_rf import CResUnetSmallRF
from train.dataset import (CROP, GAMMA_RANGE, augment_geometric, augment_photometric, compute_norm_stats,
                           load_img_and_masks, load_split, sample_crop)
from train.losses import BCEDiceLoss
from train.split import count_from_coco, split

IMG_DIR = Path("data/green/trainval/images")
MASK_DIR = Path("data/cleaned_masks/trainval/Green/masks")
COCO = Path("data/green/trainval/ground_truths/COCO/annotations_green_trainval.json")
RUN = Path("runs/cresunet_light")                # the adopted network: c-ResUNet-light
assert MASK_DIR.exists(), "run notebook 01 first: it creates the cleaned masks"

## 1. Train / validation split

80% of the 212 trainval images for training, 20% for validation. The split is **stratified by quartiles of the number of
cells** (official COCO count), so that both parts contain empty, sparse and crowded images in the same proportions.
It was created once (seed 42) and saved in `splits/green_split.json`; here it is recreated and checked.

In [ ]:
names, counts, count_of = count_from_coco(IMG_DIR, COCO)
train_names, val_names = split(names, counts, val_size=0.2, seed=42)
saved = json.load(open("splits/green_split.json"))
assert train_names == saved["train"] and val_names == saved["val"], "the split differs from splits/green_split.json"
print(f"split recreated, identical to splits/green_split.json: {len(train_names)} train, {len(val_names)} validation images")
print(pd.DataFrame({"train": pd.Series([count_of[n] for n in train_names]).describe(),
                    "validation": pd.Series([count_of[n] for n in val_names]).describe()}).round(1).T.to_string())

The two parts have almost the same distribution of cells per image (quartiles 6 / 12 / 22 and 6.5 / 12 / 22.5).

## 2. Loading and normalization

All training images are loaded once into memory (~1 GB). The pixel values are divided by 255 and normalized with **one
mean and one standard deviation over the three channels**, computed on the training images only (notebook 01, section 5,
explains why not per channel). The same values are stored in the checkpoint and reused at inference.

In [ ]:
t = time.time()
images, masks = load_img_and_masks(IMG_DIR, MASK_DIR, train_names)
mean, std = compute_norm_stats(images)
ckpt = torch.load(RUN / "best.pt", map_location="cpu", weights_only=False)
assert float(mean) == ckpt["mean"] and float(std) == ckpt["std"], "normalization differs from the checkpoint"
print(f"{len(images)} images loaded in {time.time() - t:.0f} s | mean {float(mean):.4f}, std {float(std):.4f} (identical to the checkpoint)")
print(f"after normalization the values go from {(0 - mean) / std:.2f} (black) to {(1 - mean) / std:.2f} (white)")
print(f"cell pixels: {100 * masks.mean():.2f}% of all pixels")

## 3. Training samples: random crops and augmentation

The network is trained on **random 512 × 512 crops** (uniform position, uniform image). Three crops out of four contain cells
(notebook 01), so there is no need to sample around the cells. One "epoch" is 1200 crops, about one pass over all the
training pixels (a crop covers 13.6% of an image).

Every crop is augmented on the fly:
- **geometry:** one of the 8 symmetries of the square (horizontal flip × rotations of 90°), applied to image and mask: cells
  have no preferred orientation, and no interpolation is needed, so the outlines stay exact;
- **brightness:** a random gain in [0.75, 1.5] (the brightness range between images, notebook 01, section 13);
- **contrast:** a random gamma in [0.8, 1.25].

Below: one crop and five random augmentations of it (same brightness scale for all panels; annotated cells outlined).

In [ ]:
def outline(ax, mask):
    '''Draw the outline of the cells of a mask.'''
    for contour in measure.find_contours(np.asarray(mask) > 0, 0.5):
        ax.plot(contour[:, 1], contour[:, 0], color="magenta", linewidth=1)

# a random crop with at least 6 cells, for a clearer figure
torch.manual_seed(3)
n_cells = 0
while n_cells < 6:
    crop, crop_mask = sample_crop(images, masks)
    n_cells = measure.label(crop_mask).max()
crop = crop.astype(np.float32) / 255

fig, axes = plt.subplots(1, 6, figsize=(21, 3.8))
axes[0].imshow(np.clip(crop / 0.6, 0, 1))
outline(axes[0], crop_mask)
axes[0].set_title("original crop")
for ax in axes[1:]:
    augmented, augmented_mask = augment_geometric(crop, crop_mask)
    augmented, gain, gamma = augment_photometric(augmented, return_params=True)
    ax.imshow(np.clip(augmented / 0.6, 0, 1))
    outline(ax, augmented_mask)
    ax.set_title(f"gain {gain:.2f}, gamma {gamma:.2f}")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 4. The model: c-ResUNet-light

The c-ResUNet (Morelli et al., 2021) is the architecture proposed by the authors of the dataset: an encoder–decoder with skip
connections (U-Net family), built from residual blocks. The adopted network, **c-ResUNet-light**, is the same architecture
without one block: the residual block with 5 × 5 kernels at the end of the encoder (see below and section 7).

- **Stem:** a 1 × 1 convolution from 3 channels to 1 learns the colour combination (for Green it weighs the G channel), then
  two 3 × 3 convolutions.
- **Encoder:** two levels, each a 2 × 2 max-pooling followed by a residual block (BatchNorm → ELU → convolution, twice, plus a
  shortcut): 16 → 32 → 64 channels.
- **Bottleneck:** one more pooling and one residual block: 128 channels at 1/8 of the resolution. The reference c-ResUNet adds
  here a second residual block with 5 × 5 kernels, to see further without another pooling; that block alone has 0.82 M
  parameters, 62% of the reference model.
- **Decoder:** three levels, each a 2 × 2 transposed convolution (×2 resolution), concatenation with the encoder features of
  the same resolution (skip connection, which brings back the fine detail needed for the outlines) and a residual block.
- **Head:** a 1 × 1 convolution giving one **logit** per pixel; the probability is `sigmoid(logit)`.

The original model (`models/c_res_unet.py`) ends with a sigmoid; the version used for training (`CResUnetLogits`) only
replaces the head so that it returns logits (needed for the numerically stable `BCEWithLogits` loss and for mixed precision).
`CResUnetSmallRF` (`models/c_res_unet_small_rf.py`) is the same model without the 5 × 5 block.

In [ ]:
model = CResUnetSmallRF()
model.load_state_dict(ckpt["model"])
model.eval()
n_params = sum(p.numel() for p in model.parameters())
assert n_params == 504_631

# a full image through the blocks one by one, to see the size of each output
x = torch.zeros(1, 3, 1200, 1600)
with torch.no_grad():
    s1 = model.stem(x)
    s2 = model.enc2(s1)
    s3 = model.enc3(s2)
    b = model.bottleneck(s3)
    d3 = model.dec3(b, s3)
    d2 = model.dec2(d3, s2)
    d1 = model.dec1(d2, s1)
    h = model.head(d1)

rows = []
for name, output in [("stem", s1), ("enc2", s2), ("enc3", s3), ("bottleneck", b),
                     ("dec3", d3), ("dec2", d2), ("dec1", d1), ("head", h)]:
    block = getattr(model, name)
    rows.append({"block": name,
                 "output (channels, H, W)": tuple(output.shape[1:]),
                 "parameters": sum(p.numel() for p in block.parameters())})
print(pd.DataFrame(rows).to_string(index=False))

# time for one full image on CPU
start = time.time()
with torch.no_grad():
    model(x)
print(f"\ntotal parameters: {n_params:,} | one full 1200 x 1600 image on CPU: {time.time() - start:.1f} s")

0.50 million parameters, 46% of them in the bottleneck (the reference c-ResUNet has 1.32 million, 79% in the bottleneck).
1200 and 1600 are divisible by 8 (three poolings), so the whole image goes through the network at once, with no tiling.

**Receptive field.** Each output pixel depends only on a square window of the input: about **96 × 96 px** for this network,
counting layer by layer (about 160 px with the 5 × 5 block). We can check it: give the network a random image and look at
which input pixels change the output pixel at the centre (the pixels with a non-zero gradient).

In [ ]:
torch.manual_seed(0)
x = torch.randn(1, 3, 512, 512, requires_grad=True)
output = model(x)
output[0, 0, 256, 256].backward()                 # gradient of the central output pixel

changes_output = x.grad.abs().sum(dim=1)[0] > 0   # input pixels that influence it
rows = torch.nonzero(changes_output)[:, 0]
print("receptive field:", int(rows.max() - rows.min() + 1), "px")

92 px for this pixel (between 92 and 100 px, depending on where the pixel falls with respect to the pooling windows).
The window is larger than the largest cell (68 px): to decide whether a pixel belongs to a cell, the network sees the whole
cell, its border and the background around it. The reference c-ResUNet sees about 160 px; on these images the larger
window brings no measurable gain (section 7).

## 5. The loss: BCE + soft Dice

Only 0.6% of the pixels are cell pixels. The loss is the sum, with equal weights, of:
- **binary cross-entropy (BCE)** per pixel: stable gradients, but dominated by the background;
- **1 − soft Dice** over the whole batch: Dice computed on probabilities (differentiable), insensitive to the amount of background.

Toy example: a 100 × 100 image with a 60-pixel cell, and a "lazy" prediction that says background everywhere.

In [ ]:
criterion = BCEDiceLoss()

# a 100 x 100 image with one cell of 60 pixels
target = torch.zeros(1, 1, 100, 100)
target[0, 0, :6, :10] = 1

# three predictions, as logits (large positive = cell, large negative = background)
perfect = torch.where(target == 1, 20.0, -20.0)
lazy = torch.full_like(target, -5.0)        # background everywhere
wrong = -perfect

for name, logits in [("perfect", perfect), ("lazy (all background)", lazy), ("completely wrong", wrong)]:
    loss, bce, dice = criterion(logits, target)
    print(f"{name:22s} loss {loss:7.3f} | BCE {bce:7.3f} | soft Dice {dice:.3f}")

The lazy prediction has an almost perfect BCE (0.04) but a soft Dice close to 0 (0.01), so the total loss stays high: the Dice term
forces the network to find the cells.

## 6. Training

| setting | value | why |
|---|---|---|
| optimizer | AdamW, learning rate 1e-3, weight decay 1e-4 | standard for U-Nets; not tuned (no overfitting to correct) |
| schedule | cosine from 1e-3 to 0 over 100 epochs, no early stopping | the last epochs, with a small learning rate, are the most stable |
| batch | 8 crops of 512 × 512 | fits the T4 memory; enough for BatchNorm |
| epoch | 1200 crops | about one pass over the training pixels |
| mixed precision | float16 on GPU (the loss in float32) | ~2× faster on a T4 |
| seed | 42 | reproducibility |
| checkpoint | the epoch with the best global validation Dice | Dice is the segmentation objective |

The command below retrains the network (GPU needed: ~1 h on a Kaggle T4, `kaggle/train.ipynb`). By default it is not run: the saved log of the training is shown instead.

In [ ]:
RUN_TRAINING = False
command = [sys.executable, "-m", "train.training", "--images_dir", str(IMG_DIR), "--masks_dir", str(MASK_DIR),
           "--split_json", "splits/green_split.json", "--out_dir", "runs/cresunet_light_retrained",
           "--small_rf", "--epochs", "100", "--patience", "100"]
print(" ".join(command[1:]))
if RUN_TRAINING:
    import subprocess
    subprocess.run(command, check=True)

config = json.load(open(RUN / "config.json"))
print({k: config[k] for k in ["small_rf", "epochs", "batch_size", "crops_per_epoch", "lr", "weight_decay", "patience", "seed", "device", "amp"]})

In [ ]:
log = pd.read_csv(RUN / "log.csv")
best = log.loc[log.val_dice_global.idxmax()]
assert int(best.epoch) == ckpt["epoch"] == 78

fig, axes = plt.subplots(1, 3, figsize=(17, 3.8))
axes[0].plot(log.epoch, log.train_loss, label="train (8 augmented crops)")
axes[0].plot(log.epoch, log.val_loss, label="validation (full images)")
axes[0].set_title("loss (absolute values not comparable)")
axes[0].legend()
axes[1].plot(log.epoch, log.train_dice, label="train, soft Dice")
axes[1].plot(log.epoch, log.val_dice_global, label="validation, Dice")
axes[1].axvline(best.epoch, color="grey", linestyle=":")
axes[1].set_title("Dice")
axes[1].legend()
axes[2].plot(log.epoch, log.lr)
axes[2].set_title("learning rate")
for ax in axes:
    ax.set_xlabel("epoch")
plt.tight_layout()
plt.show()

last = log.tail(10)
print(f"best epoch {int(best.epoch)}: validation Dice {best.val_dice_global:.4f} (the saved checkpoint)")
print(f"last 10 epochs: validation Dice {last.val_dice_global.mean():.4f} ± {last.val_dice_global.std():.4f}")
print(f"last 10 epochs: train soft Dice {last.train_dice.mean():.4f}")
print(f"training time: {log.time_s.sum() / 60:.0f} min ({log.time_s.mean():.0f} s per epoch, Kaggle T4)")

- The validation Dice rises quickly in the first 20 epochs and is almost flat after epoch ~60; the checkpoint is epoch 78
  (validation Dice 0.762).
- **No overfitting:** at the end the Dice on the training crops (0.77) is about the same as on the validation images (0.76),
  and the validation loss does not rise. More epochs do not help (section 7).
- The network does not even reproduce the labels it was trained on much better than new ones. Notebook 03 shows why: the
  remaining errors are mostly faint cells where the annotations themselves are ambiguous.

## 7. Tried and not adopted

Changes to the architecture and to the training recipe, each tested against the adopted network with everything else
unchanged (same seed) and compared on the **43 validation images** (network + post-processing; difference from the adopted
network, 95% bootstrap interval over images). A change was adopted only if the interval of the object F1 was entirely
above 0, with Dice and count error not demonstrably worse. None passed.

| variant | what changes | result (difference from the adopted network) | decision |
|---|---|---|---|
| reference c-ResUNet | with the 5 × 5 block of the bottleneck: receptive field ~160 px instead of ~96, 1.32 M parameters (2.6×) | F1 [−0.020, +0.006], Dice [−0.007, +0.009]; training images: F1 [−0.006, +0.010]; 5-fold cross-validation: mean validation Dice 0.749 vs 0.747 | not adopted: no demonstrable gain for 2.6× the parameters |
| early stopping | stop after 15 epochs without improvement (stopped at epoch 52, learning rate still high) | F1 [−0.034, −0.001]: demonstrably worse | full cosine schedule kept |
| no gamma | brightness augmentation without the contrast (gamma) part | F1 [−0.013, +0.009] | not adopted: no demonstrable difference |
| per-image normalization | each image normalized with the statistics of its own background | F1 [−0.037, +0.005]; count error [+0.05, +1.26], demonstrably worse | not adopted |
| wider network | 32 base channels instead of 16 (2.0 M parameters, 4×) | F1 [−0.011, +0.019], precision [+0.006, +0.035]; training images: F1 [−0.014, +0.004], Dice demonstrably lower | not adopted: capacity is not the limit |
| 200 epochs | twice as long | F1 [−0.005, +0.016]; training Dice rises to 0.78, validation does not | not adopted |
| weight maps (Morelli et al., 2021) | BCE weighted more on the pixels between close cells, to separate touching cells | **5-fold cross-validation** (5 networks per recipe, 212 images), whole pipeline: F1 [−0.001, +0.010]; merged objects 32 vs 32 | not adopted: no demonstrable effect |

Details and the cross-validation: `experiments/`. The runs were executed on Kaggle with `kaggle/train.ipynb` (adopted and
reference network), `kaggle/recipe_variants.ipynb` (one option per variant) and `kaggle/cross_validation.ipynb` (the 10
trainings of the cross-validation).

## 8. Summary

The adopted network is `runs/cresunet_light/best.pt`: the c-ResUNet-light trained for 100 epochs on 169 images, checkpoint
of epoch 78 (validation Dice 0.762). Notebook 03 turns its probability maps into individual cells.